[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabakrish/Data-Science-Teaching/blob/main/Student_Performance_Analysis.ipynb)

# Student Performance Analysis

**Question:** How do weekly study hours, attendance and prior assessment score relate to a student's final score, and how well can we predict it?

**Data:** `student_performance_synthetic.csv` is a **synthetic practice dataset** (200 students, 6 columns) created for teaching. Results show the method, not real student behaviour.

| Column | Meaning |
| --- | --- |
| `student_id` | Student identifier (1-200) |
| `weekly_study_hours` | Hours of study per week |
| `attendance_percent` | Attendance, 0-100 |
| `prior_assessment_score` | Score on an earlier assessment, 0-100 |
| `final_score` | Final score, 0-100 (the outcome we model) |
| `study_time_category` | Low / Moderate / High study time |

**Workflow:** load and check the data, explore it, fit a linear regression with a train/test split and cross-validation, then fit a logistic regression for pass/fail.

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score,
                             roc_auc_score, RocCurveDisplay)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42  # makes the split and cross-validation repeatable

## 2. Load the data

Put `student_performance_synthetic.csv` in the same folder as this notebook (or in a `data/` folder next to it). In Google Colab, upload it with the Files panel. The code below looks in a few common places so the notebook does not depend on one machine's path.

In [ ]:
CANDIDATES = [
    Path("student_performance_synthetic.csv"),
    Path("data/student_performance_synthetic.csv"),
    Path("Data_student_performance_analysis/student_performance_synthetic.csv"),
    Path("/content/student_performance_synthetic.csv"),
    Path("/content/sample_data/Data_student_performance_analysis/student_performance_synthetic.csv"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "student_performance_synthetic.csv not found. Place it next to this notebook "
        "or add its location to CANDIDATES above.")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
df.head()

## 3. Check the data

In [ ]:
print("Shape:", df.shape)
print()
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())
df.describe().round(2)

In [ ]:
# Basic quality checks. The notebook stops with an error if a check fails.
assert df["student_id"].is_unique, "student_id values should be unique"
for col in ["attendance_percent", "prior_assessment_score", "final_score"]:
    assert df[col].between(0, 100).all(), f"{col} should be between 0 and 100"
assert df["weekly_study_hours"].ge(0).all(), "weekly_study_hours should not be negative"
print("All data checks passed.")

## 4. Explore the data

### 4.1 Distributions

In [ ]:
cols = ["weekly_study_hours", "attendance_percent", "prior_assessment_score", "final_score"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, cols):
    sns.histplot(df[col], bins=15, kde=True, ax=ax)
    ax.set_title(col.replace("_", " ").title())
plt.tight_layout()
plt.show()

### 4.2 Which variables move with the final score?

In [ ]:
predictors = ["weekly_study_hours", "attendance_percent", "prior_assessment_score"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, predictors):
    sns.regplot(data=df, x=col, y="final_score", ax=ax,
                scatter_kws={"alpha": 0.5}, line_kws={"color": "crimson"})
    r = df[col].corr(df["final_score"])
    ax.set_title(f"{col.replace('_', ' ').title()} (r = {r:.2f})")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()

### 4.3 Final score by study-time category

In [ ]:
order = ["Low", "Moderate", "High"]
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="study_time_category", y="final_score", order=order)
plt.title("Final score by study-time category")
plt.show()

df.groupby("study_time_category")[["weekly_study_hours", "final_score"]].agg(["count", "mean", "std"]).round(2).loc[order]

## 5. Linear regression: predict the final score

We hold out 25% of the students as a **test set** the model never sees while fitting. A model that only memorises the training data would look good there and poor on the test set.

In [ ]:
features = ["weekly_study_hours", "attendance_percent", "prior_assessment_score"]
X = df[features]
y = df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

lin = LinearRegression().fit(X_train, y_train)
pred = lin.predict(X_test)

# Baseline: always predict the training mean. A useful model must beat this.
baseline_pred = np.full(len(y_test), y_train.mean())

def regression_report(name, y_true, y_hat):
    return {"model": name,
            "R2": r2_score(y_true, y_hat),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_hat)),
            "MAE": mean_absolute_error(y_true, y_hat)}

pd.DataFrame([regression_report("Mean baseline", y_test, baseline_pred),
              regression_report("Linear regression", y_test, pred)]).round(3)

### 5.1 Cross-validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_r2 = cross_val_score(LinearRegression(), X, y, cv=cv, scoring="r2")
cv_rmse = -cross_val_score(LinearRegression(), X, y, cv=cv, scoring="neg_root_mean_squared_error")
print("5-fold R2  :", cv_r2.round(3), "-> mean", cv_r2.mean().round(3))
print("5-fold RMSE:", cv_rmse.round(2), "-> mean", cv_rmse.mean().round(2))

### 5.2 Interpret the coefficients

Each coefficient is the change in the predicted final score for a one-unit increase in that variable, holding the others fixed. Standardised coefficients (per one standard deviation) put the variables on a common scale so their sizes can be compared.

In [ ]:
coef = pd.DataFrame({"feature": features, "coefficient": lin.coef_})
scaled = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
coef["standardised"] = scaled[-1].coef_
print("Intercept:", round(lin.intercept_, 3))
coef.round(3)

In [ ]:
# Optional: confidence intervals and p-values (needs statsmodels).
try:
    import statsmodels.api as sm
    ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    print(ols.summary().tables[1])
except ImportError:
    print("statsmodels is not installed; run `pip install statsmodels` to see p-values and intervals.")

### 5.3 Check the residuals

In [ ]:
resid = y_test - pred
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(pred, resid, alpha=0.6)
axes[0].axhline(0, color="crimson")
axes[0].set(xlabel="Predicted final score", ylabel="Residual", title="Residuals vs predicted")
sns.histplot(resid, bins=15, kde=True, ax=axes[1])
axes[1].set(xlabel="Residual", title="Residual distribution")
plt.tight_layout()
plt.show()

**What to look for:** residuals scattered evenly around zero with no curve or funnel shape suggest a linear model is reasonable here. A pattern would suggest a missing variable or a non-linear relationship.

## 6. Logistic regression: predict pass or fail

To show classification we turn the final score into two classes. **The pass mark below (60) is an assumption chosen for teaching**; change `PASS_MARK` to see how the result moves.

In [ ]:
PASS_MARK = 60
df["passed"] = (df["final_score"] >= PASS_MARK).astype(int)
print(df["passed"].value_counts().rename({1: "pass", 0: "fail"}))
print("Pass rate:", round(df["passed"].mean(), 3))

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, df["passed"], test_size=0.25, random_state=RANDOM_STATE, stratify=df["passed"])

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_train, yc_train)
yc_pred = logit.predict(Xc_test)
yc_prob = logit.predict_proba(Xc_test)[:, 1]

# Majority-class baseline: always predict the most common class.
majority = int(yc_train.mean() >= 0.5)
baseline_acc = accuracy_score(yc_test, np.full(len(yc_test), majority))

print(f"Baseline accuracy (always predict the majority class): {baseline_acc:.3f}")
print(f"Model accuracy : {accuracy_score(yc_test, yc_pred):.3f}")
print(f"Precision      : {precision_score(yc_test, yc_pred, zero_division=0):.3f}")
print(f"Recall         : {recall_score(yc_test, yc_pred, zero_division=0):.3f}")
print(f"ROC AUC        : {roc_auc_score(yc_test, yc_prob):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(confusion_matrix(yc_test, yc_pred),
                       display_labels=["Fail", "Pass"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix (test set)")
RocCurveDisplay.from_predictions(yc_test, yc_prob, ax=axes[1])
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.show()

In [ ]:
# Odds ratios per one standard deviation increase in each predictor.
odds = pd.DataFrame({
    "feature": features,
    "odds_ratio_per_SD": np.exp(logit[-1].coef_[0]),
})
odds.round(3)

An odds ratio above 1 means a higher value of that variable raises the odds of passing, holding the others fixed; below 1 lowers them.

## 7. Conclusions

*Complete this section after you run the notebook, using your own results.*

1. **Which variables are most strongly related to the final score?** (Use the correlations, the standardised coefficients and the odds ratios.)
2. **How well does the linear model predict?** Report test R² and RMSE next to the baseline, and say whether cross-validation agrees with the test-set result.
3. **How well does the logistic model classify?** Compare accuracy with the majority-class baseline, and describe the types of errors in the confusion matrix.
4. **What does the study-time category show?** Does the box plot match the numeric relationships?

## 8. Limitations

- The data is **synthetic**, so these results illustrate the method, not real students.
- A strong association is **not proof of cause**: more study hours may go with higher scores for other reasons.
- With 200 rows and one train/test split, the test metrics are noisy; cross-validation gives a steadier estimate.
- Linear and logistic models assume a simple, additive relationship. Check the residual plots before trusting them.

## 9. Student tasks

1. Change `PASS_MARK` to 50 and 70. What happens to accuracy, precision and recall, and why?
2. Add `study_time_category` as a feature (one-hot encode it) and see whether test RMSE improves.
3. Add an interaction term such as `weekly_study_hours * attendance_percent`. Does it help?
4. Try a different `RANDOM_STATE`. How much do the test metrics change?